# Vietnam Egg Price Intelligence — Complete Preprocessing

This notebook prepares `MASTER_EGG_PRICE_DATA` for:

- **Streamlit** descriptive analytics
- **Looker / Looker Studio** dashboards
- **future forecasting** preparation (SARIMAX / XGBoost)

### Important approach

The master database remains the **source-of-truth**. This notebook does **not overwrite the raw master file**.
It creates a separate cleaned analytics dataset and quality-control outputs.

The workflow is:

`MASTER data → clean → validate → add dashboard fields → export analytics data`

## 1. Import the libraries

In [ ]:
# Standard libraries
from pathlib import Path
import re
import warnings
# Data analysis libraries
import numpy as np
import pandas as pd
# Nice table display inside Jupyter
from IPython.display import display
warnings.filterwarnings("ignore")

# Show more columns when inspecting the dataframe
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 2. Find the master database

For the cleanest local workflow, keep the MASTER file in the same project folder as this notebook **locally**.

Example local project:

```text
egg_price_project_vietnam/
├── app.py
├── preprocessing_clean_analytics.ipynb
├── MASTER_EGG_PRICE_DATA.csv          # local source-of-truth
├── ANALYTICS_EGG_PRICE_DATA.csv       # generated clean analytics file
└── requirements.txt
```

The notebook will automatically look for common MASTER filenames in the current project folder.

If your MASTER file is stored somewhere else, set `MASTER_FILE` manually in the next cell using a full path, for example:

```python
MASTER_FILE = Path(r"C:\path\to\MASTER_EGG_PRICE_DATA.csv")
```

The MASTER file is **read only** by this notebook. It is never overwritten.


In [ ]:

# ============================================================
# MASTER FILE LOCATION
# ============================================================

# Project folder used by this notebook.
# In PyCharm/Jupyter this should normally be the repository folder.
PROJECT_DIR = Path.cwd()

# OPTIONAL:
# If your MASTER file is stored somewhere else, replace None with
# the full path, for example:
#
# MASTER_FILE = Path(r"C:\path\to\MASTER_EGG_PRICE_DATA.csv")
#
MASTER_FILE = None

# Common MASTER filenames to search for in the project folder.
MASTER_CANDIDATES = [
    PROJECT_DIR / "MASTER_EGG_PRICE_DATA.csv",
    PROJECT_DIR / "MASTER_EGG_PRICE_DATA.xlsx",
    PROJECT_DIR / "MASTER_EGG_PRICE_DATA(3).xlsx",
]

# If no manual path was provided, use the first MASTER file found.
if MASTER_FILE is None:
    existing_master_files = [path for path in MASTER_CANDIDATES if path.exists()]

    if not existing_master_files:
        raise FileNotFoundError(
            "MASTER_EGG_PRICE_DATA was not found in the project folder.\n\n"
            f"Current project folder: {PROJECT_DIR}\n\n"
            "Either:\n"
            "1. copy the MASTER file into this folder, or\n"
            "2. set MASTER_FILE manually to its full path in this cell."
        )

    MASTER_FILE = existing_master_files[0]

MASTER_FILE = Path(MASTER_FILE)

if not MASTER_FILE.exists():
    raise FileNotFoundError(f"MASTER file does not exist: {MASTER_FILE}")

print("Project folder:", PROJECT_DIR)
print("MASTER file:", MASTER_FILE)


## 3. Load the master database

In [ ]:
# Read Excel or CSV automatically.
if MASTER_FILE.suffix.lower() in [".xlsx", ".xls"]:
    df_raw = pd.read_excel(MASTER_FILE)
elif MASTER_FILE.suffix.lower() == ".csv":
    df_raw = pd.read_csv(MASTER_FILE)
else:
    raise ValueError("Use an .xlsx, .xls, or .csv master file.")

# Work on a copy so the original dataframe remains untouched.
df = df_raw.copy()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
display(df.head())

## 4. Check the structure before cleaning

In [ ]:
# These are the key fields expected by the current project structure.
required_columns = [
    "Record ID",
    "Date",
    "Scrape Timestamp",
    "Source",
    "Source Type",
    "Market Level",
    "Country",
    "Region",
    "Province",
    "City",
    "Location",
    "Store Name",
    "Egg Type Raw",
    "Egg Type Normalized",
    "Brand",
    "Product Name",
    "Pack Size",
    "Egg Count",
    "Pack Price VND",
    "Price Per Egg VND",
    "Unit Raw",
    "Unit Normalized",
    "Availability",
    "Source URL",
    "Quality Flag",
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    print("WARNING - missing expected columns:")
    print(missing_required)
else:
    print("All expected core columns are present.")

print("\nColumn names:")
print(df.columns.tolist())

## 5. Standardize blank values and text

Web-scraped and historical files can contain extra spaces or text versions of missing values such as `"nan"`, `"None"`, or blank strings.

We clean those first, but **do not delete any rows**.

In [ ]:
# Text values that should be treated as missing.
missing_text_values = {"", "nan", "none", "null", "n/a", "na"}

for col in df.columns:
    if df[col].dtype == "object":
        # Remove leading/trailing spaces.
        df[col] = df[col].astype("string").str.strip()

        # Convert common text placeholders into proper missing values.
        df[col] = df[col].mask(df[col].str.lower().isin(missing_text_values), pd.NA)

print("Text cleanup completed.")

## 6. Parse dates correctly

The current master data contains more than one date style, for example:

- `8/24/2026`
- `2026-08-25`

For dashboards and time-series analysis, dates must be real date values rather than text.

We keep the original `Date` and create `Date Clean`.

In [ ]:
def parse_mixed_datetime(series):
    """Parse mixed date formats safely."""
    try:
        # pandas 2.x supports format='mixed'
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        # Fallback for older pandas versions
        return pd.to_datetime(series, errors="coerce")

# Parse observation date.
df["Date Clean"] = parse_mixed_datetime(df["Date"]).dt.normalize()

# Parse scraper timestamp separately.
# The live scrapers use Vietnam timezone offsets such as +07:00.
# Keep the local clock time but remove timezone metadata so Excel can store it safely.
scrape_ts = parse_mixed_datetime(df["Scrape Timestamp"])
try:
    if scrape_ts.dt.tz is not None:
        scrape_ts = scrape_ts.dt.tz_localize(None)
except (AttributeError, TypeError):
    pass
df["Scrape Timestamp Clean"] = scrape_ts

print("Invalid observation dates:", int(df["Date Clean"].isna().sum()))
print("Valid observation dates:", int(df["Date Clean"].notna().sum()))
print("Date range:", df["Date Clean"].min(), "to", df["Date Clean"].max())

## 7. Convert numeric fields to numbers

In [ ]:
# Numeric fields used by the dashboard now or by forecasting later.
numeric_columns = [
    "Egg Count",
    "Buying Price VND",
    "Selling Price VND",
    "Pack Price VND",
    "Price Per Egg VND",
    "Quantity Sold",
    "Stock Quantity",
    "Feed Cost VND",
]

for col in numeric_columns:
    if col in df.columns:
        # Remove commas in case a number was saved as text such as "70,900".
        if df[col].dtype == "object" or str(df[col].dtype).startswith("string"):
            df[col] = df[col].str.replace(",", "", regex=False)
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric conversion completed.")

## 8. Normalize important dashboard categories

We do not overwrite the original source fields. We create standardized versions for analysis.

In [ ]:
# ---- Price level ----
# Keep market, farmgate, and retail separate.
df["Price Level"] = (
    df["Market Level"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "market": "Market",
        "farmgate": "Farmgate",
        "retail": "Retail",
    })
)

# ---- Region ----
region_map = {
    "north": "North",
    "central": "Central",
    "south": "South",
    "unknown": "Unknown",
}

df["Region Normalized"] = (
    df["Region"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(region_map)
    .fillna(df["Region"].astype("string").str.strip().str.title())
)

# ---- Data origin ----
# Source Type currently mixes data origin and price level.
# This new field makes the meaning clearer without changing the original column.
def classify_data_origin(row):
    source_type = str(row.get("Source Type", "")).strip().lower()
    has_scrape_timestamp = pd.notna(row.get("Scrape Timestamp Clean"))

    if source_type == "historical_database":
        return "Historical Database"
    if source_type == "historical_backfill":
        return "Historical Backfill"
    if source_type == "compiled_database":
        return "Compiled Database"
    if source_type in {"market", "retail"} or has_scrape_timestamp:
        return "Daily Scraper"
    return "Other"

df["Data Origin"] = df.apply(classify_data_origin, axis=1)

print("Price levels:")
display(df["Price Level"].value_counts(dropna=False).rename("Rows").to_frame())

print("Data origins:")
display(df["Data Origin"].value_counts(dropna=False).rename("Rows").to_frame())

## 9. Normalize geography

The master file currently contains some equivalent locations written in different ways, for example:

- `Hanoi` and `Hà Nội`
- `Da Nang` and `Đà Nẵng`
- `Hue`, `Thua Thien Hue`, and `Thừa Thiên-Huế`
- `Quy Nhon` is a city in `Bình Định`

We create `Province Normalized` and `City Normalized` while keeping the original fields.

The dictionary can be expanded later if a new scraper introduces another spelling.

In [ ]:
# Province aliases observed in the project data.
province_map = {
    "Hanoi": "Hà Nội",
    "Ha Noi": "Hà Nội",
    "Hà Nội": "Hà Nội",
    "Da Nang": "Đà Nẵng",
    "Đà Nẵng": "Đà Nẵng",
    "Hue": "Thừa Thiên-Huế",
    "Thua Thien Hue": "Thừa Thiên-Huế",
    "Thừa Thiên-Huế": "Thừa Thiên-Huế",
    "Quy Nhon": "Bình Định",
    "Binh Dinh": "Bình Định",
    "Bình Định": "Bình Định",
    "Dak Lak": "Đắk Lắk",
    "Đắk Lắk": "Đắk Lắk",
    "Dong Nai": "Đồng Nai",
    "Đồng Nai": "Đồng Nai",
    "Vinh Long": "Vĩnh Long",
    "Vĩnh Long": "Vĩnh Long",
    "Tay Ninh": "Tây Ninh",
    "Tây Ninh": "Tây Ninh",
    "Thai Nguyen": "Thái Nguyên",
    "Thái Nguyên": "Thái Nguyên",
    "Lam Dong": "Lâm Đồng",
    "Lâm Đồng": "Lâm Đồng",
    "Hai Phong": "Hải Phòng",
    "Hải Phòng": "Hải Phòng",
    "Cao Bang": "Cao Bằng",
    "Cao Bằng": "Cao Bằng",
    "Lang Son": "Lạng Sơn",
    "Lạng Sơn": "Lạng Sơn",
    "Quang Ninh": "Quảng Ninh",
    "Quảng Ninh": "Quảng Ninh",
    "Bac Ninh": "Bắc Ninh",
    "Bắc Ninh": "Bắc Ninh",
    "Tuyen Quang": "Tuyên Quang",
    "Tuyên Quang": "Tuyên Quang",
    "Ha Tinh": "Hà Tĩnh",
    "Hà Tĩnh": "Hà Tĩnh",
    "Tra Vinh": "Trà Vinh",
    "Trà Vinh": "Trà Vinh",
    "Hung Yen": "Hưng Yên",
    "Hưng Yên": "Hưng Yên",
    "Quang Binh": "Quảng Bình",
    "Quảng Bình": "Quảng Bình",
    "Nghe An": "Nghệ An",
    "Nghệ An": "Nghệ An",
    "Ca Mau": "Cà Mau",
    "Cà Mau": "Cà Mau",
    "Can Tho": "Cần Thơ",
    "Cần Thơ": "Cần Thơ",
    "Quang Nam": "Quảng Nam",
    "Quảng Nam": "Quảng Nam",
    "Ben Tre": "Bến Tre",
    "Bến Tre": "Bến Tre",
    "Lai Chau": "Lai Châu",
    "Lai Châu": "Lai Châu",
    "Quang Ngai": "Quảng Ngãi",
    "Quảng Ngãi": "Quảng Ngãi",
    "Khanh Hoa": "Khánh Hòa",
    "Khánh Hòa": "Khánh Hòa",
    "Quang Tri": "Quảng Trị",
    "Quảng Trị": "Quảng Trị",
    "Soc Trang": "Sóc Trăng",
    "Sóc Trăng": "Sóc Trăng",
    "Nam Dinh": "Nam Định",
    "Nam Định": "Nam Định",
    "Thanh Hoa": "Thanh Hóa",
    "Thanh Hóa": "Thanh Hóa",
    "Vinh Phuc": "Vĩnh Phúc",
    "Vĩnh Phúc": "Vĩnh Phúc",
    "An Giang": "An Giang",
    "Ho Chi Minh City": "Ho Chi Minh City",
    "Gia Lai": "Gia Lai",
    "Hau Giang": "Hậu Giang",
    "Hậu Giang": "Hậu Giang",
    "Ha Nam": "Hà Nam",
    "Hà Nam": "Hà Nam",
    "Thai Binh": "Thái Bình",
    "Thái Bình": "Thái Bình",
}

# Copy the original province, then replace known aliases.
df["Province Normalized"] = df["Province"].replace(province_map)

# If Province is missing but a city has a reliable province mapping, use it.
city_to_province = {
    "Hanoi": "Hà Nội",
    "Hà Nội": "Hà Nội",
    "Da Nang": "Đà Nẵng",
    "Hue": "Thừa Thiên-Huế",
    "Thừa Thiên-Huế": "Thừa Thiên-Huế",
    "Quy Nhon": "Bình Định",
    "Ho Chi Minh City": "Ho Chi Minh City",
}

province_from_city = df["City"].replace(city_to_province)
province_from_city = province_from_city.where(df["City"].isin(city_to_province.keys()))
df["Province Normalized"] = df["Province Normalized"].fillna(province_from_city)

# Standardize city spelling separately.
city_map = {
    "Hanoi": "Hà Nội",
    "Ha Noi": "Hà Nội",
    "Hà Nội": "Hà Nội",
    "Da Nang": "Đà Nẵng",
    "Đà Nẵng": "Đà Nẵng",
    "Hue": "Huế",
    "Quy Nhon": "Quy Nhơn",
    "Can Tho": "Cần Thơ",
    "Cần Thơ": "Cần Thơ",
    "Long Xuyen": "Long Xuyên",
}

df["City Normalized"] = df["City"].replace(city_map)

print("Geography normalization completed.")

## 10. Clean and validate prices

For comparisons across sources, `Price Per Egg VND` is the main normalized analytical price.

We create `Price Per Egg VND Clean` and protect the dashboard from obvious parser errors.

### Known example
A WinMart product named `Ba Vì 729 Omega 3` was previously interpreted as **729 eggs** because the parser picked up `729` from the product name.

`729` is part of the product/brand name, not the number of eggs in the package.

Because the actual pack quantity is not reliably available in that record, preprocessing should **not guess** the egg count.

The workflow therefore:

1. keeps the original record in the MASTER database;
2. flags the row as an `ERROR`;
3. keeps the problem row in `DATA_QUALITY_ISSUES.csv`;
4. excludes the row from `ANALYTICS_EGG_PRICE_DATA.csv`.

This keeps the dashboard dataset clean without destroying the original source record.


In [ ]:
# Keep clean analytical copies.
df["Egg Count Clean"] = df["Egg Count"].copy()
df["Price Per Egg VND Clean"] = df["Price Per Egg VND"].copy()

# Broad QC limits. These are intentionally wide and can be adjusted later.
MIN_REASONABLE_PRICE_PER_EGG = 500
MAX_REASONABLE_PRICE_PER_EGG = 20_000
MAX_REASONABLE_RETAIL_EGG_COUNT = 60

# Suspicious retail pack-size parser result.
suspect_pack = (
    df["Price Level"].eq("Retail")
    & (
        (df["Egg Count Clean"] <= 0)
        | (df["Egg Count Clean"] > MAX_REASONABLE_RETAIL_EGG_COUNT)
    )
)

# Do not allow suspicious pack counts to contaminate per-egg analytics.
df.loc[suspect_pack, "Egg Count Clean"] = np.nan
df.loc[suspect_pack, "Price Per Egg VND Clean"] = np.nan

# If normalized retail price is missing but pack price and a valid egg count exist,
# calculate the per-egg price.
can_calculate_retail = (
    df["Price Level"].eq("Retail")
    & df["Price Per Egg VND Clean"].isna()
    & df["Pack Price VND"].notna()
    & df["Egg Count Clean"].gt(0)
)

df.loc[can_calculate_retail, "Price Per Egg VND Clean"] = (
    df.loc[can_calculate_retail, "Pack Price VND"]
    / df.loc[can_calculate_retail, "Egg Count Clean"]
)

# Mark obviously unreasonable normalized prices.
suspect_price = (
    df["Price Per Egg VND Clean"].notna()
    & (
        (df["Price Per Egg VND Clean"] < MIN_REASONABLE_PRICE_PER_EGG)
        | (df["Price Per Egg VND Clean"] > MAX_REASONABLE_PRICE_PER_EGG)
    )
)

# Protect dashboard averages from these values.
df.loc[suspect_price, "Price Per Egg VND Clean"] = np.nan

print("Suspicious retail pack sizes:", int(suspect_pack.sum()))
print("Suspicious price values removed from analytics:", int(suspect_price.sum()))

## 11. FeedIn unit validation switch

FeedIn currently contains a potential unit-definition issue: some records preserve `Unit Raw = VND/kg` while the normalized analytical field is `VND/egg`.

Do **not** silently change the raw source value.

Set the switch below to `True` only after the source's egg-price unit has been confirmed and documented.

In [ ]:
# Change to True only after FeedIn's published unit is confirmed/documented.
FEEDIN_UNIT_CONFIRMED = False

feedin_unit_mismatch = (
    df["Source"].eq("FeedIn")
    & df["Unit Raw"].astype("string").str.lower().eq("vnd/kg")
    & df["Unit Normalized"].astype("string").str.lower().eq("vnd/egg")
)

print("FeedIn rows with VND/kg -> VND/egg normalization:", int(feedin_unit_mismatch.sum()))
print("FEEDIN_UNIT_CONFIRMED =", FEEDIN_UNIT_CONFIRMED)

## 12. Duplicate checks

Two similar-looking records may represent different stores, products, or source observations, so we first flag duplicate groups for review.

For the final dashboard dataset, however, an **extra exact analytical duplicate copy** should not be counted twice.

Therefore:

- the first copy is kept;
- additional exact duplicate copies are excluded from `ANALYTICS_EGG_PRICE_DATA.csv`;
- duplicate groups remain visible in `DATA_QUALITY_ISSUES.csv` for traceability.


In [ ]:
# Use a detailed observation key.
duplicate_key = [
    "Date Clean",
    "Source",
    "Price Level",
    "Province Normalized",
    "City Normalized",
    "Location",
    "Store Name",
    "Egg Type Normalized",
    "Brand",
    "Product Name",
    "Pack Size",
    "Price Per Egg VND",
]

# Only use columns that actually exist.
duplicate_key = [col for col in duplicate_key if col in df.columns]

df["Duplicate Flag"] = np.where(
    df.duplicated(subset=duplicate_key, keep=False),
    "REVIEW",
    "OK",
)

print("Rows included in possible duplicate groups:", int((df["Duplicate Flag"] == "REVIEW").sum()))

## 13. Build one quality status and issue list

The original `Quality Flag` is preserved.

For dashboards, we add two simpler fields:

- `Quality Status`: `OK`, `REVIEW`, or `ERROR`
- `Preprocessing Issues`: explains why a row needs attention

In [ ]:
def base_quality_status(value):
    text = "" if pd.isna(value) else str(value).upper()
    if text.startswith("ERROR"):
        return "ERROR"
    if text.startswith("REVIEW"):
        return "REVIEW"
    if text.startswith("OK"):
        return "OK"
    return "REVIEW"


df["Quality Status"] = df["Quality Flag"].apply(base_quality_status)
df["Preprocessing Issues"] = ""


def add_issue(mask, issue, status="REVIEW"):
    """Append a readable issue label and update Quality Status."""
    global df
    mask = mask.fillna(False)

    current = df.loc[mask, "Preprocessing Issues"].astype("string")
    df.loc[mask, "Preprocessing Issues"] = np.where(
        current.eq("") | current.isna(),
        issue,
        current + "; " + issue,
    )

    if status == "ERROR":
        df.loc[mask, "Quality Status"] = "ERROR"
    elif status == "REVIEW":
        # Do not downgrade an existing ERROR back to REVIEW.
        reviewable = mask & ~df["Quality Status"].eq("ERROR")
        df.loc[reviewable, "Quality Status"] = "REVIEW"


# Core data checks.
add_issue(df["Date Clean"].isna(), "INVALID_OR_MISSING_DATE", "ERROR")
add_issue(df["Price Per Egg VND"].isna(), "MISSING_PRICE_PER_EGG", "REVIEW")
add_issue(df["Price Level"].isna(), "MISSING_PRICE_LEVEL", "REVIEW")
add_issue(suspect_pack, "SUSPECT_RETAIL_PACK_SIZE", "ERROR")
add_issue(suspect_price, "PRICE_OUTSIDE_QC_RANGE", "REVIEW")
add_issue(df["Duplicate Flag"].eq("REVIEW"), "POSSIBLE_DUPLICATE", "REVIEW")

# Require at least one useful geographic/location field.
missing_location = (
    df["Province Normalized"].isna()
    & df["City Normalized"].isna()
    & df["Location"].isna()
    & df["Store Name"].isna()
)
add_issue(missing_location, "MISSING_LOCATION", "REVIEW")

# Keep FeedIn available, but visibly flag the unit issue until confirmed.
if not FEEDIN_UNIT_CONFIRMED:
    add_issue(feedin_unit_mismatch, "FEEDIN_UNIT_NEEDS_CONFIRMATION", "REVIEW")

# Clean empty issue strings.
df["Preprocessing Issues"] = df["Preprocessing Issues"].replace("", pd.NA)

print("Quality status after preprocessing:")
display(df["Quality Status"].value_counts(dropna=False).rename("Rows").to_frame())

## 14. Add time fields for Streamlit and Looker

These fields make filtering and grouping much easier.

In [ ]:
# Calendar fields.
df["Year"] = df["Date Clean"].dt.year.astype("Int64")
df["Month Number"] = df["Date Clean"].dt.month.astype("Int64")
df["Month"] = df["Date Clean"].dt.month_name()
df["Year Month"] = df["Date Clean"].dt.to_period("M").astype("string")
df["Quarter"] = "Q" + df["Date Clean"].dt.quarter.astype("Int64").astype("string")
df["Week"] = df["Date Clean"].dt.isocalendar().week.astype("Int64")
df["Day of Week"] = df["Date Clean"].dt.day_name()

# Simple field for whether an observation has an analytical price.
df["Analytics Price Available"] = np.where(
    df["Price Per Egg VND Clean"].notna(), "Yes", "No"
)

print("Dashboard time fields added.")

## 15. Build the clean analytics dataset and validate 2026 coverage

The MASTER database remains unchanged.

Here we create a dashboard-ready copy.

A row is excluded from `ANALYTICS_EGG_PRICE_DATA.csv` when:

- `Quality Status = ERROR`;
- the cleaned date is missing/invalid;
- the cleaned per-egg price is missing;
- it is an additional exact duplicate copy.

`REVIEW` rows are **not automatically deleted** because some still contain valid analytical prices. Their quality status remains visible so they can be monitored.


In [ ]:
# ------------------------------------------------------------
# Create a full processed copy first.
# This still contains every MASTER row for QA / traceability.
# ------------------------------------------------------------
df_processed = df.sort_values(
    by=["Date Clean", "Source", "Price Level"],
    na_position="last",
).reset_index(drop=True)

# ------------------------------------------------------------
# Define the basic eligibility rules for dashboard analytics.
# ------------------------------------------------------------
basic_eligible = (
    ~df_processed["Quality Status"].eq("ERROR")
    & df_processed["Date Clean"].notna()
    & df_processed["Price Per Egg VND Clean"].notna()
)

# ------------------------------------------------------------
# Remove only EXTRA exact duplicate copies.
# The first occurrence is retained.
# ------------------------------------------------------------
duplicate_extra = pd.Series(False, index=df_processed.index)

if duplicate_key:
    duplicate_extra.loc[basic_eligible] = (
        df_processed.loc[basic_eligible]
        .duplicated(subset=duplicate_key, keep="first")
        .to_numpy()
    )

# ------------------------------------------------------------
# Create transparent inclusion / exclusion fields.
# These remain useful when reviewing preprocessing behavior.
# ------------------------------------------------------------
df_processed["Analytics Eligible"] = "Yes"
df_processed["Analytics Exclusion Reason"] = pd.NA

error_mask = df_processed["Quality Status"].eq("ERROR")
invalid_date_mask = df_processed["Date Clean"].isna()
missing_clean_price_mask = df_processed["Price Per Egg VND Clean"].isna()

df_processed.loc[error_mask, "Analytics Eligible"] = "No"
df_processed.loc[error_mask, "Analytics Exclusion Reason"] = "QUALITY_ERROR"

df_processed.loc[
    ~error_mask & invalid_date_mask,
    "Analytics Eligible"
] = "No"
df_processed.loc[
    ~error_mask & invalid_date_mask,
    "Analytics Exclusion Reason"
] = "INVALID_OR_MISSING_DATE"

df_processed.loc[
    ~error_mask & ~invalid_date_mask & missing_clean_price_mask,
    "Analytics Eligible"
] = "No"
df_processed.loc[
    ~error_mask & ~invalid_date_mask & missing_clean_price_mask,
    "Analytics Exclusion Reason"
] = "NO_VALID_CLEAN_PRICE"

df_processed.loc[duplicate_extra, "Analytics Eligible"] = "No"
df_processed.loc[duplicate_extra, "Analytics Exclusion Reason"] = "DUPLICATE_COPY"

# ------------------------------------------------------------
# QUALITY FILE:
# Keep REVIEW and ERROR rows for investigation.
# This is NOT the dashboard input.
# ------------------------------------------------------------
df_quality_issues = df_processed[
    df_processed["Quality Status"].isin(["REVIEW", "ERROR"])
].copy()

# ------------------------------------------------------------
# ANALYTICS FILE:
# Only rows that are safe to use in charts and KPIs.
# ------------------------------------------------------------
df_analytics = df_processed[
    df_processed["Analytics Eligible"].eq("Yes")
].copy().reset_index(drop=True)

print(f"MASTER / processed rows: {len(df_processed):,}")
print(f"Clean analytics rows:    {len(df_analytics):,}")
print(f"Rows excluded:           {len(df_processed) - len(df_analytics):,}")
print(f"Extra duplicates removed:{int(duplicate_extra.sum()):,}")

# ------------------------------------------------------------
# 2026 monthly coverage based on CLEAN analytics data only.
# ------------------------------------------------------------
df_2026 = df_analytics[df_analytics["Year"].eq(2026)].copy()

monthly_2026 = (
    df_2026.groupby(["Month Number", "Month"], dropna=False)
    .agg(
        Rows=("Record ID", "size"),
        Unique_Dates=("Date Clean", "nunique"),
        Sources=("Source", "nunique"),
        Price_Levels=("Price Level", "nunique"),
        Valid_Prices=("Price Per Egg VND Clean", "count"),
        Review_Rows=("Quality Status", lambda s: int((s == "REVIEW").sum())),
    )
    .reset_index()
    .sort_values("Month Number")
)

display(monthly_2026)


## 16. Source and price-level coverage checks

In [ ]:
source_summary = (
    df_analytics.groupby(["Source", "Price Level", "Data Origin"], dropna=False)
    .agg(
        Rows=("Record ID", "size"),
        First_Date=("Date Clean", "min"),
        Last_Date=("Date Clean", "max"),
        Unique_Dates=("Date Clean", "nunique"),
        Valid_Prices=("Price Per Egg VND Clean", "count"),
    )
    .reset_index()
    .sort_values("Rows", ascending=False)
)

display(source_summary.head(10))


## 17. Inspect the final analytics dataframe

`df_analytics` is now the **clean dashboard dataset**.

It does not contain hard-error rows, invalid dates, missing clean analytical prices, or extra exact duplicate copies.

For dashboard calculations, use:

- `Date Clean`
- `Price Level`
- `Region Normalized`
- `Province Normalized`
- `City Normalized`
- `Price Per Egg VND Clean`
- `Quality Status`

`REVIEW` rows can remain because review does not necessarily mean the observation is wrong. It means the record needs monitoring or further source validation.


In [ ]:
print(f"Analytics rows: {len(df_analytics):,}")
print(f"Analytics columns: {len(df_analytics.columns):,}")
print(
    "Rows with valid clean price:",
    f"{df_analytics['Price Per Egg VND Clean'].notna().sum():,}"
)

print(
    "ERROR rows remaining in analytics:",
    int(df_analytics["Quality Status"].eq("ERROR").sum())
)

display(df_analytics.head())


## 18. Review rows that need attention

This file is separate from the dashboard data.

`DATA_QUALITY_ISSUES.csv` contains `REVIEW` and `ERROR` records so that scraper/source problems can be investigated without contaminating dashboard KPIs.

The original observations are also still preserved in the MASTER database.


In [ ]:
quality_issue_summary = (
    df_quality_issues["Preprocessing Issues"]
    .fillna("SOURCE_QUALITY_FLAG_REVIEW")
    .value_counts()
    .rename_axis("Issue")
    .reset_index(name="Rows")
)

print(f"Rows requiring review/error: {len(df_quality_issues):,}")
display(quality_issue_summary.head(30))



## 19. Export clean CSV files

The main analytics file is written directly to the **project root**, beside `app.py`:

`ANALYTICS_EGG_PRICE_DATA.csv`

This is the single dataset used by both:

- **Streamlit**
- **Looker / Looker Studio**

Quality-control CSVs are kept separately inside a `quality_reports` folder so the GitHub project root stays clean.

The MASTER database is **not overwritten**.


In [ ]:

# ============================================================
# EXPORT PROCESSED DATA - CSV FILES ONLY
# ============================================================

# The main analytics file goes directly into the project root.
analytics_path = PROJECT_DIR / "ANALYTICS_EGG_PRICE_DATA.csv"

# Keep QA/supporting outputs in a separate folder.
QA_DIR = PROJECT_DIR / "quality_reports"
QA_DIR.mkdir(exist_ok=True)

quality_path = QA_DIR / "DATA_QUALITY_ISSUES.csv"
monthly_path = QA_DIR / "MONTHLY_COVERAGE_2026.csv"
source_path = QA_DIR / "SOURCE_COVERAGE_SUMMARY.csv"
issue_summary_path = QA_DIR / "QUALITY_ISSUE_SUMMARY.csv"

# ------------------------------------------------------------
# 1. Main clean analytics dataset
# Used by both Streamlit and Looker.
# ------------------------------------------------------------
df_analytics.to_csv(
    analytics_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 2. Records that need quality review
# ------------------------------------------------------------
df_quality_issues.to_csv(
    quality_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 3. Monthly coverage for 2026
# ------------------------------------------------------------
monthly_2026.to_csv(
    monthly_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. Coverage by data source
# ------------------------------------------------------------
source_summary.to_csv(
    source_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. Summary of preprocessing / quality issues
# ------------------------------------------------------------
quality_issue_summary.to_csv(
    issue_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Files created successfully:\n")
print("MAIN DASHBOARD DATA:")
print(" -", analytics_path.resolve())

print("\nQUALITY REPORTS:")
for path in [
    quality_path,
    monthly_path,
    source_path,
    issue_summary_path,
]:
    print(" -", path.resolve())


## 20. Final sanity checks

Run this cell every time the MASTER file is refreshed.

For dashboard use, the final analytics file should have:

- no `ERROR` rows;
- no missing cleaned observation dates;
- no missing `Price Per Egg VND Clean`;
- no additional exact duplicate copies;
- no suspicious retail pack counts above the QC threshold.

Problem rows remain available in the MASTER and quality-control output.


In [ ]:
print("FINAL PREPROCESSING CHECK")
print("=" * 55)

print(f"Raw MASTER rows:                 {len(df_raw):,}")
print(f"Processed rows before filtering: {len(df_processed):,}")
print(f"Clean analytics rows:            {len(df_analytics):,}")
print(f"Rows excluded from analytics:    {len(df_processed) - len(df_analytics):,}")

print("\nCLEAN ANALYTICS VALIDATION")
print("-" * 55)
print(f"ERROR rows remaining:            {(df_analytics['Quality Status'] == 'ERROR').sum():,}")
print(f"Invalid/missing dates:           {df_analytics['Date Clean'].isna().sum():,}")
print(f"Missing clean prices:            {df_analytics['Price Per Egg VND Clean'].isna().sum():,}")

remaining_duplicate_extras = 0
if duplicate_key:
    remaining_duplicate_extras = int(
        df_analytics.duplicated(subset=duplicate_key, keep="first").sum()
    )

print(f"Extra exact duplicates:          {remaining_duplicate_extras:,}")

bad_retail_pack = (
    df_analytics["Price Level"].eq("Retail")
    & df_analytics["Egg Count Clean"].notna()
    & (
        (df_analytics["Egg Count Clean"] <= 0)
        | (df_analytics["Egg Count Clean"] > MAX_REASONABLE_RETAIL_EGG_COUNT)
    )
)

print(f"Suspicious retail pack counts:   {int(bad_retail_pack.sum()):,}")
print(f"2026 observations:               {(df_analytics['Year'] == 2026).sum():,}")
print(
    f"2026 months represented:         "
    f"{df_analytics.loc[df_analytics['Year'] == 2026, 'Month Number'].nunique():,}"
)
print(f"Sources represented:             {df_analytics['Source'].nunique():,}")
print(f"FeedIn unit confirmed:           {FEEDIN_UNIT_CONFIRMED}")

# ------------------------------------------------------------
# Hard safety checks.
# If any of these fail, do not use the output for the dashboard.
# ------------------------------------------------------------
assert not df_analytics["Quality Status"].eq("ERROR").any(), \
    "ERROR rows are still present in analytics data."

assert df_analytics["Date Clean"].notna().all(), \
    "Analytics data contains invalid/missing dates."

assert df_analytics["Price Per Egg VND Clean"].notna().all(), \
    "Analytics data contains missing clean prices."

assert remaining_duplicate_extras == 0, \
    "Extra exact duplicate copies are still present."

assert not bad_retail_pack.any(), \
    "Suspicious retail pack-size records are still present."

print("\nPASS - ANALYTICS_EGG_PRICE_DATA is clean for dashboard use.")



# Next step

Use:

`ANALYTICS_EGG_PRICE_DATA.csv`

as the input for both the updated Streamlit dashboard and later Looker.

Your core repository can therefore remain simple:

```text
egg_price_project_vietnam/
├── app.py
├── preprocessing_clean_analytics.ipynb
├── ANALYTICS_EGG_PRICE_DATA.csv
└── requirements.txt
```

When preprocessing is run, supporting QA CSVs are written to:

`quality_reports/`

The MASTER remains the source-of-truth, while `ANALYTICS_EGG_PRICE_DATA.csv` is regenerated after each MASTER refresh.

Recommended Streamlit filters:

1. Date range
2. Price Level — **Market / Farmgate / Retail**
3. Source
4. Region
5. Province
6. Egg Type
7. Production System
8. Brand
9. Product
10. Store
11. Quality Status

For normal dashboard KPIs, use `Price Per Egg VND Clean`.
